# Figure 4 — HOMER and TransBrain

The two methods solve related but different problems. TransBrain is a region-level phenotype
regressor built on a transcriptomic embedding; HOMER is a parcel-level probabilistic
correspondence built on connectivity. On region-level homologue identification they are evenly
matched, on TransBrain's own atlas and curation, and they separate on most other axes.

One caveat is specific to this figure: half of it depends on the third-party `transbrain`
package. The cell below reports whether it is importable. Without it the scores still recompute
from the cached per-region distributions, but the distributions themselves cannot be rebuilt.

Before running: `python scripts/fetch_data.py`, and optionally the `transbrain` package.

In [ ]:
import importlib.util, json, subprocess, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FIGS = ROOT.parent / 'manuscript' / 'figures'
sys.path.insert(0, str(ROOT / 'src'))

from homer.data import load_pi, pi_provenance

pi = load_pi()
PROV = pi_provenance()
LOGS = ROOT / 'outputs' / 'logs'
print(f"coupling {pi.shape[0]:,} x {pi.shape[1]:,}   {PROV['pi_file']}   sha {PROV['pi_sha256'][:16]}...")

try:
    import transbrain
    HAVE_TB = True
    print("transbrain: available -- the distributions can be regenerated")
except Exception as e:
    HAVE_TB = False
    print(f"transbrain: NOT available ({type(e).__name__}) -- scores recompute from cached "
          f"distributions, but the distributions themselves cannot be rebuilt")

PUBLISHED = {
    'HOMER AUROC':          (0.83, 0.01),
    'TransBrain AUROC':     (0.84, 0.01),
    'paired Wilcoxon p':    (0.36, 0.02),
    'HOMER eff. regions':   (6,    0.5),
    'TransBrain eff. regions': (60, 1.0),
    'HOMER mass on region': (0.21, 0.01),
    'TransBrain mass on region': (0.07, 0.01),
    'round-trip gradient (HOMER)': (0.97, 0.01),
    'round-trip Magel2 (HOMER)':   (0.91, 0.01),
}

def check(name, value):
    exp, tol = PUBLISHED[name]
    ok = abs(value - exp) <= tol
    print(f"  [{'ok ' if ok else 'FAIL'}] {name:30s} computed {value:.4g}   manuscript {exp}")
    return ok


def verified_log(fname):
    d = json.loads((LOGS / fname).read_text())
    sha = d.get('pi_sha256') if isinstance(d, dict) else None
    if sha is None:
        print(f"  {fname}: NO coupling provenance recorded")
    elif sha != PROV['pi_sha256']:
        raise RuntimeError(f"{fname} was built on a DIFFERENT coupling ({sha[:16]}...)")
    else:
        print(f"  {fname}: provenance verified")
    return d

## 1. Region identity (Fig. 4a)

Both methods are scored as distributions over the same 127-region Brainnetome atlas, on the same
24 literature homologue pairs. Every score below is recomputed from the per-region distributions
rather than read from the summary log.

Worth noting that this is TransBrain's own atlas and its own literature curation, so matching it
here is matching it on home ground.

In [ ]:
RUN_DUMPS = False       # True regenerates the distributions themselves (needs transbrain)

if RUN_DUMPS and HAVE_TB:
    for d in ['dump_bn_distributions.py', 'dump_roundtrip_maps.py']:
        print(f'regenerating via {d} ...')
        r = subprocess.run([sys.executable, str(FIGS / 'fig4' / d)], cwd=str(ROOT),
                           capture_output=True, text=True)
        print((r.stdout or r.stderr)[-300:])
elif RUN_DUMPS:
    print('RUN_DUMPS requested but transbrain is unavailable; skipping')

bn = verified_log('transbrain_bn_distributions.json')
# 'regions' is a dict keyed by mouse region acronym, not a list.
region_names = list(bn['regions'])
regions = [bn['regions'][k] for k in region_names]
print(f"\n{len(regions)} benchmark regions over {len(bn['bn_names'])} Brainnetome targets")


def auroc(weights, true_idx):
    '''Rank the true homologue region against all other targets.'''
    w = np.asarray(weights, float)
    y = np.zeros(len(w), bool); y[list(true_idx)] = True
    pos, neg = w[y], w[~y]
    if not len(pos) or not len(neg):
        return np.nan
    # ties count as half, matching the standard Mann-Whitney formulation
    return float((( pos[:, None] > neg[None, :]).sum()
                  + 0.5 * (pos[:, None] == neg[None, :]).sum()) / (len(pos) * len(neg)))


def eff_n(weights):
    '''Effective number of target regions, 1 / sum(p^2) -- the sharpness measure.'''
    w = np.asarray(weights, float); s = w.sum()
    return float(1.0 / np.square(w / s).sum()) if s > 0 else np.nan


h_auc = np.array([auroc(r['homer_w'], r['true']) for r in regions])
t_auc = np.array([auroc(r['tb_w'], r['true']) for r in regions])
h_eff = np.array([eff_n(r['homer_w']) for r in regions])
t_eff = np.array([eff_n(r['tb_w']) for r in regions])
h_mass = np.array([np.asarray(r['homer_w'], float)[list(r['true'])].sum() for r in regions])
t_mass = np.array([np.asarray(r['tb_w'], float)[list(r['true'])].sum() for r in regions])
p_w = float(wilcoxon(h_auc, t_auc).pvalue)

print(f"\nAUROC      HOMER {h_auc.mean():.3f}   TransBrain {t_auc.mean():.3f}")
print(f"           TransBrain leads in {int((t_auc > h_auc).sum())} of {len(h_auc)} regions, "
      f"paired Wilcoxon p = {p_w:.3f}")
print(f"sharpness  HOMER {h_eff.mean():.1f}   TransBrain {t_eff.mean():.1f}  effective target regions")
print(f"mass       HOMER {h_mass.mean():.3f}   TransBrain {t_mass.mean():.3f}  on the correct region\n")
check('HOMER AUROC', float(h_auc.mean()))
check('TransBrain AUROC', float(t_auc.mean()))
check('paired Wilcoxon p', p_w)
check('HOMER eff. regions', float(h_eff.mean()))
check('TransBrain eff. regions', float(t_eff.mean()))
check('HOMER mass on region', float(h_mass.mean()))
check('TransBrain mass on region', float(t_mass.mean()))

print("\nEqual accuracy, delivered differently: TransBrain reaches the right region with a")
print("diffuse estimate spread over ~60 targets, HOMER with a committed one over ~6.")

## 2. Round-trip fidelity (Fig. 4b)

Translate a mouse phenotype mouse→human→mouse and correlate the recovered map with the original.
Both methods are scored on the same 52 mouse regions, those where the phenotype is measured and
HOMER has parcels. Scoring the two on different region sets is a real hazard here: an earlier log
did exactly that and reported TransBrain ahead on the optogenetic map.

In [ ]:
rt = verified_log('transbrain_roundtrip_maps.json')

print()
for name, v in rt.items():
    if not isinstance(v, dict) or 'r_homer' not in v:
        continue
    orig = np.asarray(v['original'], float)
    lead = 'HOMER' if v['r_homer'] > v['r_transbrain'] else 'TransBrain'
    print(f"  {name:10s} HOMER {v['r_homer']:.3f}   TransBrain {v['r_transbrain']:.3f}   "
          f"({v['n_regions_scored']} regions, {lead} ahead)")

print()
check('round-trip gradient (HOMER)', rt['gradient']['r_homer'])
check('round-trip Magel2 (HOMER)', rt['Magel2']['r_homer'])

# Recompute the correlations from the stored maps rather than trusting the recorded r.
#
# THE SCORING IS REGION-LEVEL, NOT PARCEL-LEVEL. The stored maps are 1,864-parcel vectors, but
# the reported r is a Pearson correlation over the 52 MOUSE REGIONS where the phenotype is
# measured and HOMER has parcels -- the same region set for both methods. Correlating the raw
# parcel vectors gives a different (and wrong) number: 0.91 rather than 0.97 for the gradient.
# This is the same class of mistake as the 52-vs-68-region scoring bug; the aggregation is the
# result, not an implementation detail.
meta = json.loads((ROOT / 'data_external' / 'mouse_sc_meta.json').read_text())
parcel_region = np.array([meta['structure_acronyms'][i] for i in meta['node_struct_idx']])

def region_mean(vec, region_list):
    vec = np.asarray(vec, float)
    out = []
    for reg in region_list:
        m = (parcel_region == reg) & np.isfinite(vec)
        out.append(vec[m].mean() if m.any() else np.nan)
    return np.array(out)

print("\nrecomputed from the stored maps, aggregated to the scored mouse regions:")
for name, v in rt.items():
    if not isinstance(v, dict) or 'original' not in v:
        continue
    scored = v['regions_scored']
    o = region_mean(v['original'], scored)
    for meth, key in (('HOMER', 'homer'), ('TransBrain', 'transbrain')):
        if key not in v:
            raise KeyError(f'{name}: no recovered map under {key!r}')
        rec = region_mean(v[key], scored)
        m = np.isfinite(o) & np.isfinite(rec)
        rr = float(np.corrcoef(o[m], rec[m])[0, 1])
        recorded = v[f'r_{key}']
        flag = 'ok' if abs(rr - recorded) < 0.005 else 'MISMATCH'
        print(f"  {name:10s} {meth:11s} r = {rr:.3f}   recorded {recorded:.3f}   [{flag}]")

## 3. Gradient translation and the capability ledger (Fig. 4d)

Routing the mouse principal gradient forward, HOMER tracks the observed human gradient somewhat
better than TransBrain over the 101 covered Brainnetome regions. The two translated gradients
agree closely with each other, which suggests both recover the same signal at different fidelity.

Panel d's table is derived rather than typed. Its accuracy row is graded from the Wilcoxon result
computed in §1: where the difference is not significant, both methods are graded full. That row
was previously hand-graded partial/full on a difference the paper itself calls n.s.

In [ ]:
bench = verified_log('transbrain_2025_benchmark.json')
g = bench['head_to_head_gradient']
print(f"\ngradient over {g['n_bn_regions']} Brainnetome regions:")
print(f"  HOMER      vs observed  r = {g['homer_vs_reference']:.3f}")
print(f"  TransBrain vs observed  r = {g['transbrain_vs_reference']:.3f}")
print(f"  the two methods agree with each other at r = {g['homer_vs_transbrain']:.3f}")

print(f"\ncapability ledger, as panel d derives it:")
print(f"  accuracy    {h_auc.mean():.2f} vs {t_auc.mean():.2f}  -> "
      f"{'full/full (n.s.)' if p_w >= 0.05 else 'full/partial'}")
print(f"  gradient    {g['homer_vs_reference']:.2f} vs {g['transbrain_vs_reference']:.2f}")
print(f"  resolution  {pi.shape[1]:,} parcels vs ~{len(bn['bn_names'])} regions")
print(f"  sharpness   ~{h_eff.mean():.0f} vs ~{t_eff.mean():.0f} effective targets")
rh = [v['r_homer'] for v in rt.values() if isinstance(v, dict) and 'r_homer' in v]
rtb = [v['r_transbrain'] for v in rt.values() if isinstance(v, dict) and 'r_transbrain' in v]
print(f"  round-trip  {min(rh):.2f}-{max(rh):.2f} vs {min(rtb):.2f}-{max(rtb):.2f}")

## 4. Build the figure panels

In [ ]:
RUN_PANELS = True

PANELS = [('a', 'make_panelA_accuracy.py'), ('b', 'make_panelB_roundtrip.py'),
          ('c', 'make_panelC_candidates.py'), ('d', 'make_panelD_matrix.py')]

if RUN_PANELS:
    for panel, script in PANELS:
        r = subprocess.run([sys.executable, str(FIGS / 'fig4' / script)], cwd=str(ROOT),
                           capture_output=True, text=True)
        print(f"panel {panel}  {script:28s} {'ok' if r.returncode == 0 else 'FAILED'}")
        if r.returncode != 0:
            print((r.stderr or '')[-400:])
else:
    print('skipped; set RUN_PANELS = True')

## 5. Summary

Equivalent region-level accuracy on TransBrain's own benchmark, delivered an order of magnitude
more sharply; better gradient translation and round-trip fidelity; parcel rather than region
resolution. Across the seven capability axes compared, HOMER leads on six and is level on the
seventh.

The two are probably better read as complementary instruments, region-level phenotype transfer
against calibrated whole-brain correspondence, than as competitors.

In [ ]:
print(f"coupling            {PROV['pi_file']}")
print(f"AUROC               {h_auc.mean():.2f} vs {t_auc.mean():.2f} (Wilcoxon p = {p_w:.2f})")
print(f"sharpness           {h_eff.mean():.0f} vs {t_eff.mean():.0f} effective target regions")
print(f"round-trip          {min(rh):.2f}-{max(rh):.2f} vs {min(rtb):.2f}-{max(rtb):.2f}")
print(f"gradient            {g['homer_vs_reference']:.2f} vs {g['transbrain_vs_reference']:.2f}")
print()
ok = all([check('HOMER AUROC', float(h_auc.mean())),
          check('TransBrain AUROC', float(t_auc.mean())),
          check('paired Wilcoxon p', p_w),
          check('HOMER eff. regions', float(h_eff.mean())),
          check('TransBrain eff. regions', float(t_eff.mean())),
          check('round-trip gradient (HOMER)', rt['gradient']['r_homer']),
          check('round-trip Magel2 (HOMER)', rt['Magel2']['r_homer'])])
print('\nALL CHECKS PASS' if ok else '\nSOME CHECKS FAILED -- text and code have diverged')